### Dataset and Task Metadata

In [9]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="SRBCT",
    dataset_year="2001",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="OpenML",
    original_dataset_source_download_link="https://www.openml.org/d/45101",
    download_description="""
We download the data from OpenML to a predefined folder.

mkdir -p local-data-warehouse/SRBCT/ && wget -P local-data-warehouse/SRBCT/ https://api.openml.org/data/download/22112162/dataset 
""",
    # References
    academic_reference_bibtex="""@article{Khan2001ClassificationAD,
  title={Classification and diagnostic prediction of cancers using gene expression profiling and artificial neural networks},
  author={Javed Khan and Jun S. Wei and Markus Ringnér and Lao H. Saal and Marc Ladanyi and Frank Westermann and Frank Berthold and Manfred Schwab and Cristina R. Antonescu and Carsten Peterson and Paul S. Meltzer},
  journal={Nature Medicine},
  year={2001},
  volume={7},
  number={6},
  pages={673-679},
  doi={10.1038/89044},
  url={https://doi.org/10.1038/89044}
}
""",
    academic_reference_bibtex_key="Khan2001ClassificationAD",
    license="Public Domain",
    data_tags=["IID"],
    curation_comments="""
        - We use the OpenML version of the dataset (training and test excluding 5 non-SRBCT test samples) as the original reference does not mention any official data repostitory. 
        - We use target feature encoding as in the plsgenomics R package (https://rdrr.io/cran/plsgenomics/man/SRBCT.html).
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="CancerType",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="CancerType",
)

## Preprocessing

In [10]:
import pandas as pd
import arff

with open(dataset_mold.path/"dataset") as f:
    data = arff.load(f)
df = pd.DataFrame(data["data"], columns=[x[0] for x in data["attributes"]])

target_feature = "CancerType"
df = df.rename(columns={"CLASS": target_feature})
df[target_feature] = df[target_feature].map({"1": "EWS", "2": "BL", "3": "NB", "4": "RMS"}).astype("category")

In [11]:
df.columns

Index(['gene1', 'gene2', 'gene3', 'gene4', 'gene5', 'gene6', 'gene7', 'gene8',
       'gene9', 'gene10',
       ...
       'gene2300', 'gene2301', 'gene2302', 'gene2303', 'gene2304', 'gene2305',
       'gene2306', 'gene2307', 'gene2308', 'CancerType'],
      dtype='object', length=2309)

## Data Checks

In [12]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 83
Columns: 2309
Use sampling: False (sample size: 83)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['gene1154', 'gene1955', 'gene1177', 'gene1956', 'gene1175', 'gene1174', 'gene1173', 'gene1957', 'gene1958', 'gene1170']
Rows remaining as candidates after top-10 filter: 0 (of 83)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [13]:
# Sample Rows
df_head

gene1   gene2   gene3   gene4   gene5   gene6   gene7   gene8   gene9  \
0  3.2025  0.0681  1.0460  0.1243  0.4941  3.1207  3.7106  1.8416  1.2607   
1  1.6547  0.0710  1.0409  0.0520  0.2045  2.1609  2.4452  1.1473  0.7371   
2  3.2779  0.1160  0.8926  0.1014  0.2818  1.9773  3.2590  1.4106  0.9548   
3  1.0060  0.1906  0.4302  0.1035  0.2984  1.6804  5.8901  0.2958  0.7381   
4  2.7098  0.2367  0.3693  0.2190  0.3711  1.7800  3.2376  0.6769  0.8546   

   gene10  gene11  gene12  gene13  gene14  gene15  gene16  gene17  gene18  \
0  2.9001  4.0270  1.0643  4.0651  1.4730  2.7932  0.4815  1.4482  3.3214   
1  1.9989  2.6131  0.8541  4.7284  2.4784  1.5103  0.8961  0.4850  2.3431   
2  2.0775  4.8139  0.4257  4.7120  2.7548  1.9162  1.2710  1.1331  2.4818   
3  1.6610  4.9105  1.5866  9.4802  0.1667  1.1314  2.6361  0.8405  0.9928   
4  0.6808  4.5104  0.6461  3.6433  1.7957  1.0375  0.3976  0.8846  1.5156   

   gene19  gene20  gene21  gene22  gene23  gene24  gene25  gene26  gene27  \
0  0.7022  1.7260  1.5136  3.9255  0.5296  3.9098  3.7136  4.0016  1.1819   
1  0.2531  1.7841  1.0886  5.9544  0.5337  2.7007  3.2339  3.4448  2.8296   
2  2.0350  1.7340  2.6863  5.5842  1.1332  4.6055  2.2437  3.8154  1.6393   
3  0.1239  0.5216  0.9867  4.8170  0.6451  2.0627  3.1900  3.4317  0.1925   
4  1.2582  1.0114  1.5428  5.1313  0.6248  4.4183  2.1173  5.5302  1.8151   

   gene28  gene29  gene30  gene31  gene32  gene33  gene34  gene35  gene36  \
0  2.1459  0.6662  1.7203  0.2690  1.4677  0.6380  2.1497  3.7382  2.7538   
1  0.9659  0.4829  1.3528  0.7262  1.2882  2.5171  3.0538  2.8671  1.9144   
2  1.9209  0.7984  1.5175  0.5605  1.5492  1.8893  2.0080  4.8112  2.2913   
3  0.9634  0.3900  0.3349  0.3040  0.3668  1.7941  2.7727  1.3236  0.8575   
4  0.7825  0.4637  0.7564  0.6539  0.9060  0.9905  2.6055  0.9257  1.5953   

   gene37  gene38  gene39  gene40  gene41  gene42  gene43  gene44  gene45  \
0  0.8587  0.9320  1.0597  0.2684  1.1909  3.5375  0.1115  0.8346  1.5854   
1  0.5963  0.9130  0.5030  1.3195  0.9951  3.1554  0.1146  0.6188  0.5013   
2  0.2019  1.2415  0.6952  0.3134  0.5556  2.7144  0.1014  0.4320  2.9859   
3  1.5864  0.4192  0.6228  0.1715  3.3604  1.6481  0.1132  0.5376  2.0020   
4  0.4982  0.8406  0.6296  0.3414  2.3463  1.2607  0.3376  0.4145  0.5376   

   gene46  gene47  gene48  gene49  gene50  gene51  gene52  gene53  gene54  \
0  1.6778  3.9576  3.7285  3.6545  0.3247  3.5717  0.8967  2.3148  0.3194   
1  1.6110  3.0636  1.7920  1.3757  0.3092  3.7014  0.7999  2.1661  0.3089   
2  1.5970  3.4367  2.0991  3.8527  0.4848  2.6743  0.7425  2.6301  0.3077   
3  0.9297  2.5482  5.2167  5.4350  0.3963  0.7093  0.6052  0.2447  0.1816   
4  0.8110  3.4904  2.4205  4.1405  0.1880  1.3702  0.6890  2.0653  0.2358   

   gene55  gene56  gene57  gene58  gene59  gene60  gene61  gene62  gene63  \
0  3.4188  0.8722  3.9192  0.1288  3.1675  3.3678  3.6157  3.8972  1.2217   
1  2.7758  2.0965  5.4148  0.0782  3.7242  3.6836  3.3596  7.6184  1.5961   
2  2.1161  0.9193  4.5608  0.2194  2.6364  3.7572  3.1295  5.5745  0.9068   
3  1.4857  0.4664  2.6340  0.1067  1.3142  1.2415  2.6222  9.5706  0.2286   
4  1.3889  0.4443  4.0206  0.3066  2.6916  6.1149  3.9849  3.5135  0.3605   

   gene64  gene65  gene66  gene67  gene68  gene69  gene70  gene71  gene72  \
0  3.3361  1.0708  0.3088  0.1087  0.7344  0.6572  0.1167  0.4576  0.5910   
1  2.4137  0.9393  0.4445  0.1556  0.6798  0.5171  0.1669  0.5398  0.4766   
2  3.5267  1.4182  0.7817  0.1721  1.4264  0.5040  0.1709  0.3552  0.9151   
3  2.2161  0.7152  2.6169  0.9200  4.3382  0.5790  0.2541  1.4058  0.9887   
4  1.9250  0.5949  0.9356  0.2059  0.5245  0.5513  0.2037  0.8994  1.5808   

   gene73  gene74  gene75  gene76  gene77  gene78  gene79  gene80  gene81  \
0  0.1106  0.3945  0.1479  0.9519  2.6286  0.5856  0.1956  0.6629  0.3520   
1  0.1132  0.2986  0.3997  1.6803  2.3320  0.4157  0.2604  0.5328  0.2067   
2  0.1659  0.2757  0.3022  0.5023  3.8235  0.3742  0.2725  0.4268  0.

In [14]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,CancerType,category,0.0,0.0,4.0,"EWS, RMS, NB, BL"
1,gene1,float64,0.0,0.0,83.0,"3.2025, 2.0298, 1.2705, 1.8594, 2.2313, 2.0272, 3.1013, 2.0816, 3.4636, 1.1145"
2,gene2,float64,0.0,0.0,83.0,"0.0681, 0.7067, 0.4657, 0.524, 1.9247, 0.5502, 0.391, 0.9137, 1.2855, 1.382"
3,gene3,float64,0.0,0.0,83.0,"1.046, 1.0439, 0.9344, 0.6808, 0.2943, 0.3688, 0.3937, 0.5806, 0.3355, 0.2623"
4,gene4,float64,0.0,0.0,82.0,"0.1243, 0.1016, 0.068, 0.3013, 0.1762, 0.3627, 0.2905, 0.0673, 0.0893, 0.0733"
5,gene5,float64,0.0,0.0,83.0,"0.4941, 0.2147, 0.393, 0.4011, 0.3855, 0.6133, 0.1113, 0.6856, 0.3219, 0.5906"
6,gene6,float64,0.0,0.0,83.0,"3.1207, 0.3269, 3.2629, 3.5232, 5.3998, 5.8139, 5.6969, 1.8688, 2.3853, 1.9687"
7,gene7,float64,0.0,0.0,83.0,"3.7106, 17.533, 4.8109, 7.283, 7.1398, 7.029, 5.4799, 4.0321, 4.1111, 4.3105"
8,gene8,float64,0.0,0.0,83.0,"1.8416, 2.2418, 1.5265, 0.9024, 1.2459, 0.6437, 1.538, 1.2707, 1.9877, 0.7899"
9,gene9,float64,0.0,0.0,83.0,"1.2607, 1.7367, 1.5754, 1.9208, 1.3112, 0.5307, 0.763, 1.4253, 0.9138, 1.3615"


In [15]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
gene1,83.0,1.487880,0.988884,0.0683,4.3557
gene2,83.0,0.300580,0.371274,0.0494,1.9247
gene3,83.0,1.020083,0.671267,0.1570,3.1923
gene4,83.0,0.501959,0.558527,0.0345,1.7928
gene5,83.0,0.293716,0.154303,0.0401,0.7674
gene6,83.0,1.903552,1.644205,0.2826,11.6200
gene7,83.0,5.976558,3.171152,2.1729,19.0419
gene8,83.0,1.290884,0.629206,0.2958,3.4008
gene9,83.0,1.413651,0.697047,0.5277,4.2117
gene10,83.0,1.652846,0.712888,0.2076,3.2202


In [16]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column     rank                    
CancerType 1      EWS     29  34.94
           2      RMS     25  30.12
           3       NB     18  21.69
           4       BL     11  13.25

In [17]:
# Target Distribution
target_df

,count,pct
CancerType,,
EWS,29,34.94
RMS,25,30.12
NB,18,21.69
BL,11,13.25


## Task Curation

In [18]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=20, n_splits=3, test_size=None


In [19]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
)

Using Stratified IID splits.


## Export

In [20]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019cfbce-1521-7263-894a-710ac5ce15e4
1a6d615508a1562dfabb9709c9680f9671815e7a4070d2329212a7f70823055c
